# FraudIA Claims — 03: Evaluación del Modelo

Evaluación completa del modelo entrenado: métricas, curvas, SHAP y análisis de errores.

**Prerequisito:** Ejecutar `02_modelo_fraude.ipynb` y tener los artefactos en `models/`  
**Modelos evaluados:** RandomForestClassifier + IsolationForest

## 0. Setup y carga de artefactos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT      = Path('..') if Path('../models').exists() else Path('.')
MODELS    = ROOT / 'models'
DATA      = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'data' / 'outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Cargar artefactos
rf     = joblib.load(MODELS / 'fraud_model.pkl')
isof   = joblib.load(MODELS / 'isolation_forest.pkl')
scaler = joblib.load(MODELS / 'scaler.pkl')
cols   = json.loads((MODELS / 'model_columns.json').read_text())
mets   = json.loads((MODELS / 'metrics.json').read_text())
shap_imp = json.loads((MODELS / 'shap_feature_importance.json').read_text())

print('✓ Artefactos cargados')
print(f'  Features  : {len(cols)}')
print(f'  F1        : {mets["f1"]}')
print(f'  AUC-ROC   : {mets["auc_roc"]}')

## 1. Cargar datos y reconstruir etiquetas

In [ ]:
df = pd.read_csv(DATA / 'claims_with_documents.csv')

def create_label(row):
    if row.get('doc_factura_alterada', False):        return 1
    if row.get('doc_ruc_invalido', False):            return 1
    if row.get('proveedor_lista_restrictiva', False): return 1
    sim  = row.get('similitud_narrativa', 0) or 0
    dias = row.get('dias_desde_inicio_poliza', 999) or 999
    if sim >= 0.85 and dias <= 30:                    return 1
    if row.get('narrativa_clonada', False):           return 1
    if row.get('doc_sin_denuncia_previa', False) and row.get('doc_robo', False): return 1
    return 0

X = df[cols].copy()
for c in X.select_dtypes(include='bool').columns:
    X[c] = X[c].astype(int)
X = X.fillna(X.median(numeric_only=True)).astype(float)
y = df.apply(create_label, axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(f'Test set: {len(X_test)} siniestros  |  Sospechosos reales: {y_test.sum()}')

## 2. Métricas de clasificación

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score
)

print('=== REPORTE DE CLASIFICACIÓN ===')
print(classification_report(y_test, y_pred, target_names=['Legítimo','Sospechoso']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba):.4f}')
print(f'AP      : {average_precision_score(y_test, y_proba):.4f}')
print(f'CV F1   : {mets["cv_f1_mean"]} (±std 5-fold)')

## 3. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Legítimo','Sospechoso'], fontsize=12)
ax.set_yticks([0,1]); ax.set_yticklabels(['Legítimo','Sospechoso'], fontsize=12)
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('Matriz de Confusión — Random Forest', fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(OUTPUTS / '03_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Curva ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[0].plot(fpr, tpr, color='#1e3a5f', lw=2, label=f'AUC = {auc:.4f}')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Aleatorio')
axes[0].set_xlabel('Tasa Falsos Positivos'); axes[0].set_ylabel('Tasa Verdaderos Positivos')
axes[0].set_title('Curva ROC — Random Forest', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Curva Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1].plot(rec, prec, color='#dc2626', lw=2, label=f'AP = {ap:.4f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS / '03_curvas_roc_pr.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Importancia de features — SHAP vs RF nativo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SHAP (del archivo guardado)
shap_df = pd.DataFrame(list(shap_imp.items()), columns=['feature','shap']).sort_values('shap').tail(15)
colors_shap = ['#dc2626' if v > 0.05 else '#d97706' if v > 0.02 else '#1e3a5f' for v in shap_df['shap']]
axes[0].barh(shap_df['feature'], shap_df['shap'], color=colors_shap)
axes[0].set_title('Importancia SHAP (top 15)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia media |SHAP|')

# Feature importance nativa del RF
rf_imp = pd.Series(rf.feature_importances_, index=cols).sort_values().tail(15)
axes[1].barh(rf_imp.index, rf_imp.values, color='#3b82f6')
axes[1].set_title('Importancia RF nativa (top 15)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature Importance')

plt.tight_layout()
plt.savefig(OUTPUTS / '03_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Scores ML vs Score de Reglas

In [ ]:
model_scores = pd.read_csv(MODELS / 'model_scores.csv')
scored       = pd.read_csv(DATA / 'claims_scored.csv')
compare = model_scores.merge(scored[['id_siniestro','score_reglas','nivel_riesgo']], on='id_siniestro')

color_map = {'BAJO':'#16a34a','MEDIO':'#d97706','ALTO':'#dc2626'}
plt.figure(figsize=(9, 6))
for nivel, grp in compare.groupby('nivel_riesgo'):
    plt.scatter(grp['score_reglas'], grp['score_random_forest'],
                label=nivel, color=color_map[nivel], alpha=0.6, s=40)
plt.xlabel('Score Reglas (0-100)', fontsize=12)
plt.ylabel('Score Random Forest (0-100)', fontsize=12)
plt.title('Correlación: Score ML vs Score Reglas', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS / '03_scatter_ml_vs_reglas.png', dpi=120, bbox_inches='tight')
plt.show()

corr = compare[['score_reglas','score_random_forest','score_isolation_forest']].corr()
print('Correlación entre scores:')
print(corr.round(3))

## 7. Análisis de casos ALTO — ¿el modelo los capta bien?

In [ ]:
altos = compare[compare['nivel_riesgo'] == 'ALTO'][[
    'id_siniestro','score_reglas','score_random_forest','score_isolation_forest'
]].sort_values('score_random_forest', ascending=False)

print(f'Siniestros ALTO detectados: {len(altos)}')
print(f'Score RF medio en ALTO    : {altos["score_random_forest"].mean():.1f}')
print(f'Score IsoF medio en ALTO  : {altos["score_isolation_forest"].mean():.1f}')
print()
print(altos.to_string(index=False))

## 8. Resumen de evaluación

> **Conclusión:** El modelo RandomForest + IsolationForest identifica correctamente los casos de mayor riesgo con métricas excepcionales sobre el dataset sintético. Para producción, se recomienda validar con datos reales y monitorear el desvío de distribución.

In [ ]:
print('=== EVALUACIÓN FINAL DEL MODELO ===')
print(f'Precision        : {mets["precision"]:.3f}')
print(f'Recall           : {mets["recall"]:.3f}')
print(f'F1 Score         : {mets["f1"]:.3f}')
print(f'AUC-ROC          : {mets["auc_roc"]:.3f}')
print(f'CV F1 (5-fold)   : {mets["cv_f1_mean"]:.3f}')
print(f'Train samples    : {mets["n_train"]}')
print(f'Test samples     : {mets["n_test"]}')
print(f'Features usadas  : {mets["n_features"]}')
print()
print('Top 5 features por SHAP:')
for feat, val in list(shap_imp.items())[:5]:
    print(f'  {feat:<40s}: {val:.4f}')